# NB07 — Study-design confound analysis for genome-derived Levins B

**Motivation:** NB06 v2 shows λ=0.204 for the SPIRE genome Levins B (compared to λ≈0.80 for
MicrobeAtlas 16S Levins B in P1). Low λ suggests the genome-derived niche breadth carries little
phylogenetic signal — which is expected if Levins B is driven by *how many studies happened to
sample a genus across diverse environments* rather than true ecological niche preferences.

**Tests:**
1. Compute per-genus study diversity (n_unique_studies) and biome diversity (n_unique_biomes)
2. Show Pearson r between Levins B and these confounders
3. Estimate λ of n_unique_studies itself (is study sampling phylogenetically random?)
4. PGLS controlling for log(n_unique_studies): does the KO–niche signal survive?
5. Study-residual Levins B: regress out study count and rerun PGLS
6. Rarefied Levins B: subsample to 1 MAG per study and recompute (removes within-study replication bias)

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import dendropy
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H, grid_h
apply_style()

_REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(_REPO_ROOT / 'comprehensive_metal_ecology' / 'scripts'))
from pgls_utils import run_pgls, pgls_results_table

DATA_DIR = Path.cwd().parent / 'data'
FIGS     = Path.cwd().parent / 'figures'

TREE_PATH          = DATA_DIR / 'gtdb_bac_genus_full.tree'
MIN_MAGS_PER_GENUS = 5

print('NB07 executing.')

NB07 executing.


In [2]:
# Load feature matrix, metadata, sample biome annotations
fm       = pd.read_parquet(DATA_DIR / 'mag_feature_matrix.parquet')
meta_all = pd.read_csv(DATA_DIR / 'all_env_mag_metadata_cache.csv')
samp     = pd.read_parquet(DATA_DIR / 'spire_sample_metadata.parquet')

# Merge biome + study info onto MAG metadata
meta_biome = meta_all.merge(
    samp[['sample_id', 'study_id', 'microntology']], on='sample_id', how='left'
)
biome_df = meta_biome[
    meta_biome['microntology'].notna() & (meta_biome['microntology'] != '')
].copy()

print(f'All-env MAGs with biome: {len(biome_df):,}')
print(f'Unique studies:          {biome_df["study_id"].nunique()}')
print(f'Unique biome categories: {biome_df["microntology"].nunique()}')
print(f'Genera with biome MAGs:  {biome_df["genus"].nunique():,}')

All-env MAGs with biome: 19,198
Unique studies:          68
Unique biome categories: 37
Genera with biome MAGs:  1,773


In [3]:
# Compute per-genus diversity metrics
def levins_b(series):
    counts = series.value_counts()
    n = counts.sum()
    if n < 2:
        return np.nan
    p = counts / n
    return 1.0 / (p ** 2).sum()

genus_stats = (
    biome_df.groupby('genus')
    .agg(
        levins_b_genome  = ('microntology', levins_b),
        n_mags_biome     = ('mag_id',       'count'),
        n_unique_studies = ('study_id',      'nunique'),
        n_unique_biomes  = ('microntology',  'nunique'),
    )
    .reset_index()
)
genus_stats = genus_stats[genus_stats['n_mags_biome'] >= MIN_MAGS_PER_GENUS].copy()

print(f'Genera (n_mags >= {MIN_MAGS_PER_GENUS}): {len(genus_stats)}')
print(genus_stats[['levins_b_genome','n_mags_biome','n_unique_studies','n_unique_biomes']]
      .describe().round(3).to_string())

Genera (n_mags >= 5): 643
       levins_b_genome  n_mags_biome  n_unique_studies  n_unique_biomes
count          643.000       643.000           643.000          643.000
mean             2.689        20.855             4.675            4.252
std              1.719        29.773             4.153            3.317
min              1.000         5.000             1.000            1.000
25%              1.276         7.000             2.000            2.000
50%              2.270        11.000             3.000            3.000
75%              3.571        21.500             6.000            6.000
max             10.974       315.000            24.000           19.000


In [4]:
# Pearson correlations: Levins B vs confounders
for col, label in [
    ('n_mags_biome',    'n_mags_biome'),
    ('n_unique_studies','n_unique_studies'),
    ('n_unique_biomes', 'n_unique_biomes'),
]:
    r, p = stats.pearsonr(genus_stats['levins_b_genome'], genus_stats[col])
    rs, ps = stats.spearmanr(genus_stats['levins_b_genome'], genus_stats[col])
    print(f'levins_b ~ {label:22s}: Pearson r={r:+.3f} p={p:.2e}  |  Spearman rho={rs:+.3f} p={ps:.2e}')

levins_b ~ n_mags_biome          : Pearson r=+0.359 p=5.71e-21  |  Spearman rho=+0.424 p=1.92e-29
levins_b ~ n_unique_studies      : Pearson r=+0.802 p=2.17e-145  |  Spearman rho=+0.889 p=1.78e-219
levins_b ~ n_unique_biomes       : Pearson r=+0.862 p=1.34e-191  |  Spearman rho=+0.933 p=5.36e-287


In [5]:
# Scatter: Levins B vs n_unique_studies and n_unique_biomes
fig, axes = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

for ax, (xcol, xlabel) in zip(axes, [
    ('n_unique_studies', 'Number of unique studies'),
    ('n_unique_biomes',  'Number of unique biome categories'),
]):
    grid_h(ax)
    ax.scatter(genus_stats[xcol], genus_stats['levins_b_genome'],
               s=8, alpha=0.5, color=PALETTE[0], edgecolors='none')
    # OLS fit line
    m, b = np.polyfit(genus_stats[xcol], genus_stats['levins_b_genome'], 1)
    xr = np.linspace(genus_stats[xcol].min(), genus_stats[xcol].max(), 100)
    ax.plot(xr, m * xr + b, color='gray', lw=0.8, ls='--')
    r, p = stats.pearsonr(genus_stats['levins_b_genome'], genus_stats[xcol])
    ax.annotate(f'r={r:+.2f}, p={p:.2e}', xy=(0.05, 0.93), xycoords='axes fraction',
                fontsize=8, color='#808080')
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel('Levins B (genome biome)', fontsize=9)
    ax.set_title(f'Levins B vs {xlabel}', fontsize=10)

fig.suptitle('Study-design confound: Levins B correlates with sampling breadth', y=1.02)
save(fig, FIGS / 'nb07_levins_b_confound_scatter')
print('Saved nb07_levins_b_confound_scatter.pdf')

Saved nb07_levins_b_confound_scatter.pdf


In [6]:
# Load GTDB tree and join KO densities
gtdb_tree = dendropy.Tree.get(path=str(TREE_PATH), schema='newick', preserve_underscores=True)
tree_genera = {n.taxon.label for n in gtdb_tree.leaf_node_iter() if n.taxon}

genus_ko = (
    fm.groupby('genus')
    .agg(
        ko_per_mb_primary  = ('ko_per_mb_primary',  'median'),
        ko_per_mb_cofactor = ('ko_per_mb_cofactor', 'median'),
        n_mags_total       = ('mag_id',             'count'),
    )
    .reset_index()
)

genus_stats['genus'] = genus_stats['genus'].str.lower()
genus_ko['genus']    = genus_ko['genus'].str.lower()

feat = genus_stats.merge(genus_ko, on='genus', how='inner')
feat = feat[feat['genus'].isin(tree_genera)].copy()

feat['levins_b_genome_z']   = stats.zscore(feat['levins_b_genome'],       nan_policy='omit')
feat['ko_per_mb_primary_z'] = stats.zscore(feat['ko_per_mb_primary'],     nan_policy='omit')
feat['ko_per_mb_cofactor_z']= stats.zscore(feat['ko_per_mb_cofactor'],    nan_policy='omit')
feat['log_n_studies']       = np.log1p(feat['n_unique_studies'])
feat['log_n_studies_z']     = stats.zscore(feat['log_n_studies'],         nan_policy='omit')
feat['log_n_biomes_z']      = stats.zscore(np.log1p(feat['n_unique_biomes']), nan_policy='omit')
feat = feat.dropna(subset=['levins_b_genome_z', 'ko_per_mb_primary_z',
                            'ko_per_mb_cofactor_z', 'log_n_studies_z'])
print(f'Final n (complete cases): {len(feat)}')

# Estimate λ of n_unique_studies itself — is study sampling phylogenetically random?
feat['log_n_studies_z_clean'] = stats.zscore(feat['log_n_studies'], nan_policy='omit')
studies_pgls = run_pgls(
    df=feat, tree_path=str(TREE_PATH),
    response='log_n_studies_z',
    predictors=['ko_per_mb_primary_z'],  # dummy predictor just to get λ estimate
    taxon_col='genus',
)
studies_res = pgls_results_table([studies_pgls])
lam_studies = float(studies_res['lambda_est'].iloc[0])
print(f'λ of log(n_unique_studies): {lam_studies:.4f}')
print('  (compare: λ of Levins B in NB06 v2 = 0.2042)')
print('  If λ_studies ≈ λ_levins_b → study diversity drives the Levins B phylogenetic signal')

Final n (complete cases): 328


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


λ of log(n_unique_studies): 0.2179
  (compare: λ of Levins B in NB06 v2 = 0.2042)
  If λ_studies ≈ λ_levins_b → study diversity drives the Levins B phylogenetic signal


In [7]:
# PGLS models: with and without n_studies covariate
models = {
    'baseline (primary KO only)':          ['ko_per_mb_primary_z'],
    'baseline (cofactor KO only)':         ['ko_per_mb_cofactor_z'],
    'study-controlled (primary + studies)':['ko_per_mb_primary_z', 'log_n_studies_z'],
    'study-controlled (cofactor + studies)':['ko_per_mb_cofactor_z', 'log_n_studies_z'],
    'studies only':                        ['log_n_studies_z'],
}

model_results = []
for name, preds in models.items():
    res = run_pgls(
        df=feat, tree_path=str(TREE_PATH),
        response='levins_b_genome_z',
        predictors=preds,
        taxon_col='genus',
    )
    tbl = pgls_results_table([res])
    for _, row in tbl.iterrows():
        if row['predictor'] in preds and 'ko_per_mb' in row['predictor']:
            model_results.append({
                'model':    name,
                'predictor': row['predictor'],
                'n':        int(row['n']),
                'beta':     float(row['beta']),
                'se':       float(row['SE']),
                'p':        float(row['p_value']),
                'lambda_est': float(tbl['lambda_est'].iloc[0]),
            })
    # Also record studies-only for comparison
    if name == 'studies only':
        r0 = tbl.iloc[0]
        model_results.append({
            'model':    name,
            'predictor': 'log_n_studies_z',
            'n':        int(r0['n']),
            'beta':     float(r0['beta']),
            'se':       float(r0['SE']),
            'p':        float(r0['p_value']),
            'lambda_est': float(tbl['lambda_est'].iloc[0]),
        })
    print(f"--- {name} ---")
    print(tbl[['predictor','n','lambda_est','beta','SE','p_value']].to_string(index=False))
    print()

model_df = pd.DataFrame(model_results)
model_df.to_csv(DATA_DIR / 'nb07_study_controlled_pgls.csv', index=False)
print('\nSaved nb07_study_controlled_pgls.csv')

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


--- baseline (primary KO only) ---
          predictor   n  lambda_est      beta       SE  p_value
ko_per_mb_primary_z 328      0.2042 -0.055576 0.066977 0.407273



/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


--- baseline (cofactor KO only) ---
           predictor   n  lambda_est      beta       SE  p_value
ko_per_mb_cofactor_z 328      0.1999 -0.072315 0.062229 0.246051



/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


--- study-controlled (primary + studies) ---
          predictor   n  lambda_est     beta       SE  p_value
ko_per_mb_primary_z 328      0.0001 0.018073 0.032022 0.572864
    log_n_studies_z 328      0.0001 0.843716 0.029897 0.000000



/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


--- study-controlled (cofactor + studies) ---
           predictor   n  lambda_est      beta       SE  p_value
ko_per_mb_cofactor_z 328      0.0001 -0.007781 0.030652 0.799765
     log_n_studies_z 328      0.0001  0.841594 0.029905 0.000000



/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


--- studies only ---
      predictor   n  lambda_est     beta       SE  p_value
log_n_studies_z 328      0.0001 0.842245 0.029752      0.0


Saved nb07_study_controlled_pgls.csv


In [8]:
# Study-residual Levins B: OLS-regress out log_n_studies, run PGLS on residuals
from sklearn.linear_model import LinearRegression

lm = LinearRegression().fit(
    feat[['log_n_studies']].values,
    feat['levins_b_genome'].values,
)
feat['levins_b_study_resid'] = feat['levins_b_genome'] - lm.predict(feat[['log_n_studies']].values)
feat['levins_b_study_resid_z'] = stats.zscore(feat['levins_b_study_resid'], nan_policy='omit')

r2_study = lm.score(feat[['log_n_studies']].values, feat['levins_b_genome'].values)
print(f'OLS R² of levins_b ~ log(n_studies): {r2_study:.4f} ({r2_study*100:.1f}% variance explained by study count)')

resid_pgls_primary = run_pgls(
    df=feat.dropna(subset=['levins_b_study_resid_z', 'ko_per_mb_primary_z']),
    tree_path=str(TREE_PATH),
    response='levins_b_study_resid_z',
    predictors=['ko_per_mb_primary_z'],
    taxon_col='genus',
)
resid_tbl_primary = pgls_results_table([resid_pgls_primary])

resid_pgls_cofactor = run_pgls(
    df=feat.dropna(subset=['levins_b_study_resid_z', 'ko_per_mb_cofactor_z']),
    tree_path=str(TREE_PATH),
    response='levins_b_study_resid_z',
    predictors=['ko_per_mb_cofactor_z'],
    taxon_col='genus',
)
resid_tbl_cofactor = pgls_results_table([resid_pgls_cofactor])

print('\nResidual Levins B PGLS (study count regressed out):')
print('Primary KO:')
print(resid_tbl_primary[['predictor','n','lambda_est','beta','SE','p_value']].to_string(index=False))
print('Cofactor KO:')
print(resid_tbl_cofactor[['predictor','n','lambda_est','beta','SE','p_value']].to_string(index=False))

OLS R² of levins_b ~ log(n_studies): 0.7061 (70.6% variance explained by study count)


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(



Residual Levins B PGLS (study count regressed out):
Primary KO:
          predictor   n  lambda_est     beta       SE  p_value
ko_per_mb_primary_z 328      0.0001 0.032751 0.058754 0.577619
Cofactor KO:
           predictor   n  lambda_est      beta       SE  p_value
ko_per_mb_cofactor_z 328      0.0001 -0.014563 0.056246 0.795867


In [9]:

# Rarefied Levins B: 1 MAG per (genus, study) cell — removes within-study replication
# Shuffle then drop_duplicates — keeps 1 random MAG per genus-study pair (seed-reproducible)
biome_df_gs = biome_df[['genus', 'study_id', 'microntology']].copy()
biome_rare = (
    biome_df_gs
    .sample(frac=1, random_state=42)
    .drop_duplicates(subset=['genus', 'study_id'])
    .reset_index(drop=True)
)

genus_rare = (
    biome_rare.groupby('genus')['microntology']
    .agg(levins_b_rare=levins_b, n_mags_rare='count')
    .reset_index()
)
genus_rare = genus_rare[genus_rare['n_mags_rare'] >= MIN_MAGS_PER_GENUS].copy()
genus_rare['genus'] = genus_rare['genus'].str.lower()

print(f'MAGs before rarefaction: {len(biome_df_gs):,}')
print(f'MAGs after rarefaction (1/study): {len(biome_rare):,}')
print(f'Genera with ≥{MIN_MAGS_PER_GENUS} studies: {len(genus_rare)}')

# Correlation: rarefied vs original Levins B
comp = genus_stats[['genus', 'levins_b_genome']].merge(genus_rare, on='genus', how='inner')
r_rare, p_rare = stats.pearsonr(comp['levins_b_genome'], comp['levins_b_rare'])
print(f'Original vs rarefied Levins B: Pearson r={r_rare:.3f}, p={p_rare:.2e}, n={len(comp)}')


MAGs before rarefaction: 19,198
MAGs after rarefaction (1/study): 4,558
Genera with ≥5 studies: 230
Original vs rarefied Levins B: Pearson r=0.731, p=1.03e-39, n=230


In [10]:
# PGLS with rarefied Levins B as response
feat_rare = genus_rare.merge(genus_ko, on='genus', how='inner')
feat_rare = feat_rare[feat_rare['genus'].isin(tree_genera)].copy()
feat_rare['levins_b_rare_z']    = stats.zscore(feat_rare['levins_b_rare'],       nan_policy='omit')
feat_rare['ko_per_mb_primary_z']= stats.zscore(feat_rare['ko_per_mb_primary'],   nan_policy='omit')
feat_rare['ko_per_mb_cofactor_z']= stats.zscore(feat_rare['ko_per_mb_cofactor'], nan_policy='omit')
feat_rare = feat_rare.dropna(subset=['levins_b_rare_z', 'ko_per_mb_primary_z'])
print(f'Rarefied PGLS n: {len(feat_rare)}')

rare_pgls_primary = run_pgls(
    df=feat_rare, tree_path=str(TREE_PATH),
    response='levins_b_rare_z',
    predictors=['ko_per_mb_primary_z'],
    taxon_col='genus',
)
rare_tbl_primary = pgls_results_table([rare_pgls_primary])

rare_pgls_cofactor = run_pgls(
    df=feat_rare, tree_path=str(TREE_PATH),
    response='levins_b_rare_z',
    predictors=['ko_per_mb_cofactor_z'],
    taxon_col='genus',
)
rare_tbl_cofactor = pgls_results_table([rare_pgls_cofactor])

print('Rarefied Levins B PGLS (1 MAG per study):')
print('Primary KO:')
print(rare_tbl_primary[['predictor','n','lambda_est','beta','SE','p_value']].to_string(index=False))
print('Cofactor KO:')
print(rare_tbl_cofactor[['predictor','n','lambda_est','beta','SE','p_value']].to_string(index=False))

Rarefied PGLS n: 100


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


Rarefied Levins B PGLS (1 MAG per study):
Primary KO:
          predictor   n  lambda_est     beta       SE  p_value
ko_per_mb_primary_z 100      0.0001 0.041855 0.111574  0.70837
Cofactor KO:
           predictor   n  lambda_est     beta       SE  p_value
ko_per_mb_cofactor_z 100      0.0001 0.012506 0.105851 0.906192


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


In [11]:
# Summary forest plot: β across response variable variants
summary_rows = [
    # label, response, beta, se, p, n, lambda_est
    ('NB06 v2\n(biome Levins B, primary)', 'primary',
     -0.0556, 0.0670, 0.407, 328, 0.204),
    ('NB06 v2\n(biome Levins B, cofactor)', 'cofactor',
     -0.0723, 0.0622, 0.246, 328, 0.200),
]

# Study-controlled (from model_df)
for _, row in model_df.iterrows():
    if 'cofactor' in row['predictor'] and 'study' in row['model']:
        summary_rows.append((
            'Study-controlled\n(cofactor + log_n_studies)', 'cofactor',
            row['beta'], row['se'], row['p'], row['n'], row['lambda_est']
        ))

# Residual Levins B
r_p = resid_tbl_primary[resid_tbl_primary['predictor']=='ko_per_mb_primary_z'].iloc[0]
r_c = resid_tbl_cofactor[resid_tbl_cofactor['predictor']=='ko_per_mb_cofactor_z'].iloc[0]
summary_rows += [
    ('Study-residual Levins B\n(primary KO)', 'primary',
     float(r_p['beta']), float(r_p['SE']), float(r_p['p_value']), int(r_p['n']), float(resid_tbl_primary['lambda_est'].iloc[0])),
    ('Study-residual Levins B\n(cofactor KO)', 'cofactor',
     float(r_c['beta']), float(r_c['SE']), float(r_c['p_value']), int(r_c['n']), float(resid_tbl_cofactor['lambda_est'].iloc[0])),
]

# Rarefied
rp = rare_tbl_primary[rare_tbl_primary['predictor']=='ko_per_mb_primary_z'].iloc[0]
rc = rare_tbl_cofactor[rare_tbl_cofactor['predictor']=='ko_per_mb_cofactor_z'].iloc[0]
summary_rows += [
    ('Rarefied Levins B\n(1 MAG/study, primary)', 'primary',
     float(rp['beta']), float(rp['SE']), float(rp['p_value']), int(rp['n']), float(rare_tbl_primary['lambda_est'].iloc[0])),
    ('Rarefied Levins B\n(1 MAG/study, cofactor)', 'cofactor',
     float(rc['beta']), float(rc['SE']), float(rc['p_value']), int(rc['n']), float(rare_tbl_cofactor['lambda_est'].iloc[0])),
]

fig, ax = plt.subplots(figsize=(FIGW['2col'], ROW_H * 1.6))
ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.axvline(-0.021, color=PALETTE[3], lw=0.8, ls=':', alpha=0.6)  # P1 reference
ax.annotate('P1 ref (β=−0.021)', xy=(-0.021, len(summary_rows)-0.5),
            xytext=(-0.025, len(summary_rows)-0.8),
            fontsize=7, color=PALETTE[3], ha='right')

colors = [PALETTE[0] if k[1]=='primary' else PALETTE[2] for k in summary_rows]
for i, (lbl, _, beta, se, p, n, lam) in enumerate(summary_rows):
    y = len(summary_rows) - 1 - i
    ax.barh(y, beta, xerr=1.96 * se,
            error_kw={'elinewidth': 1.0, 'capsize': 3},
            color=colors[i], edgecolor='k', linewidth=0.5, height=0.6)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    pad = max(abs(beta) * 0.08, 0.003)
    ha  = 'left' if beta >= 0 else 'right'
    ax.text(beta + (pad if beta >= 0 else -pad), y,
            f'{sig}  n={n}  λ={lam:.2f}',
            va='center', ha=ha, fontsize=7.5, color='#808080')

ax.set_yticks(range(len(summary_rows)))
ax.set_yticklabels([r[0] for r in reversed(summary_rows)], fontsize=8)
ax.set_xlabel('PGLS β (metal gene density → niche breadth)', fontsize=9)
ax.set_title('Study-design confound sensitivity: genome-derived Levins B', fontsize=10)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=PALETTE[0], edgecolor='k', linewidth=0.5, label='Primary KO'),
    Patch(facecolor=PALETTE[2], edgecolor='k', linewidth=0.5, label='Cofactor KO'),
], fontsize=8, loc='lower right')

fig.suptitle('', y=1.02)
save(fig, FIGS / 'nb07_study_confound_sensitivity')
print('Saved nb07_study_confound_sensitivity.pdf')

Saved nb07_study_confound_sensitivity.pdf


In [12]:
# Save summary
result_summary = {
    'notebook': 'NB07_study_design_confound',
    'n_genera_eligible': len(genus_stats),
    'n_genera_pgls': len(feat),
    'lambda_levins_b_nb06v2': 0.2042,
    'lambda_log_n_studies': lam_studies,
    'r_levins_b_vs_n_studies': float(stats.pearsonr(genus_stats['levins_b_genome'], genus_stats['n_unique_studies'])[0]),
    'r_levins_b_vs_n_biomes':  float(stats.pearsonr(genus_stats['levins_b_genome'], genus_stats['n_unique_biomes'])[0]),
    'r2_levins_b_vs_log_n_studies': r2_study,
    'r_orig_vs_rarefied_levins_b': r_rare,
    'beta_primary_baseline':   -0.0556,
    'beta_cofactor_baseline':  -0.0723,
    'beta_primary_residual':   float(r_p['beta']),
    'beta_cofactor_residual':  float(r_c['beta']),
    'beta_primary_rarefied':   float(rp['beta']),
    'beta_cofactor_rarefied':  float(rc['beta']),
    'p_primary_residual':      float(r_p['p_value']),
    'p_cofactor_residual':     float(r_c['p_value']),
    'p_primary_rarefied':      float(rp['p_value']),
    'p_cofactor_rarefied':     float(rc['p_value']),
    'lambda_residual_primary': float(resid_tbl_primary['lambda_est'].iloc[0]),
    'lambda_rarefied_primary': float(rare_tbl_primary['lambda_est'].iloc[0]),
}

with open(DATA_DIR / 'nb07_study_confound_summary.json', 'w') as f:
    json.dump(result_summary, f, indent=2)

print(json.dumps(result_summary, indent=2))

{
  "notebook": "NB07_study_design_confound",
  "n_genera_eligible": 643,
  "n_genera_pgls": 328,
  "lambda_levins_b_nb06v2": 0.2042,
  "lambda_log_n_studies": 0.2179,
  "r_levins_b_vs_n_studies": 0.8016915462770943,
  "r_levins_b_vs_n_biomes": 0.8623169643935388,
  "r2_levins_b_vs_log_n_studies": 0.7061102768861799,
  "r_orig_vs_rarefied_levins_b": 0.7310237780504402,
  "beta_primary_baseline": -0.0556,
  "beta_cofactor_baseline": -0.0723,
  "beta_primary_residual": 0.032751258565594944,
  "beta_cofactor_residual": -0.014562791390696923,
  "beta_primary_rarefied": 0.041855499670074625,
  "beta_cofactor_rarefied": 0.012506131443374919,
  "p_primary_residual": 0.5776190853149212,
  "p_cofactor_residual": 0.7958667963924444,
  "p_primary_rarefied": 0.7083699502400176,
  "p_cofactor_rarefied": 0.9061918190140719,
  "lambda_residual_primary": 0.0001,
  "lambda_rarefied_primary": 0.0001
}
